In [2]:
import sys
import subprocess

subprocess.check_call([
    sys.executable,
    "-m",
    "pip",
    "install",
    "shap"
])

0

In [3]:
import pandas as pd
import numpy as np
import joblib
import shap

ROOT = r"C:\Users\Acer\Documents\AuthentiScore"

FEATURES = [
    "followers",
    "following",
    "posts",
    "bio_length",
    "has_profile_picture",
    "is_private",
    "username_digit_ratio",
    "follower_following_ratio",
    "profile_completeness"
]

print("SHAP version:", shap.__version__)
print("M19 initialized successfully!")

SHAP version: 0.52.0
M19 initialized successfully!


In [4]:
model = joblib.load(
    ROOT + r"\backend\trained_models\authentiscore_rf_v1.pkl"
)

df = pd.read_csv(
    ROOT + r"\data\processed\instagram_features.csv"
)

X = df[FEATURES]

print("Model loaded!")
print("Dataset:", X.shape)

Model loaded!
Dataset: (696, 9)


In [5]:
explainer = shap.TreeExplainer(model)

print("SHAP TreeExplainer created successfully!")

SHAP TreeExplainer created successfully!


In [6]:
profile = X.iloc[[0]]

shap_values = explainer.shap_values(profile)

print("SHAP values generated!")

SHAP values generated!


In [8]:
print("SHAP output type:", type(shap_values))
print("SHAP output shape:", np.array(shap_values).shape)

SHAP output type: <class 'numpy.ndarray'>
SHAP output shape: (1, 9, 2)


In [10]:
# Get SHAP values for the FAKE class (class 1)
fake_class_shap = shap_values[0, :, 1]

# Get the profile values as a 1-dimensional array
profile_values = profile.iloc[0].values

# Create feature contribution table
contribution_df = pd.DataFrame({
    "feature": FEATURES,
    "value": profile_values,
    "shap_value": fake_class_shap
})

# Absolute SHAP value for sorting
contribution_df["abs_shap"] = contribution_df["shap_value"].abs()

# Sort by strongest contribution
contribution_df = contribution_df.sort_values(
    "abs_shap",
    ascending=False
).reset_index(drop=True)

contribution_df

,feature,value,shap_value,abs_shap
0,posts,32.00000,-0.160410,0.160410
1,followers,1000.00000,-0.142456,0.142456
2,follower_following_ratio,1.04712,-0.091680,0.091680
3,bio_length,53.00000,-0.090772,0.090772
4,username_digit_ratio,0.27000,0.083131,0.083131
5,has_profile_picture,1.00000,-0.052111,0.052111
6,following,955.00000,-0.013305,0.013305
7,is_private,0.00000,0.003108,0.003108
8,profile_completeness,1.00000,0.000000,0.000000


In [11]:
def generate_shap_explanations(contribution_df):

    explanations = []

    for _, row in contribution_df.iterrows():

        feature = row["feature"]
        shap_value = row["shap_value"]
        value = row["value"]

        if shap_value > 0:
            direction = "increases fake-profile risk"
        elif shap_value < 0:
            direction = "decreases fake-profile risk"
        else:
            direction = "has little effect on the prediction"

        explanations.append({
            "feature": feature,
            "value": value,
            "shap_value": round(float(shap_value), 4),
            "direction": direction
        })

    return explanations

In [12]:
explanations = generate_shap_explanations(
    contribution_df
)

for explanation in explanations:

    print(
        f"{explanation['feature']}: "
        f"{explanation['direction']} "
        f"(SHAP={explanation['shap_value']})"
    )

posts: decreases fake-profile risk (SHAP=-0.1604)
followers: decreases fake-profile risk (SHAP=-0.1425)
follower_following_ratio: decreases fake-profile risk (SHAP=-0.0917)
bio_length: decreases fake-profile risk (SHAP=-0.0908)
username_digit_ratio: increases fake-profile risk (SHAP=0.0831)
has_profile_picture: decreases fake-profile risk (SHAP=-0.0521)
following: decreases fake-profile risk (SHAP=-0.0133)
is_private: increases fake-profile risk (SHAP=0.0031)
profile_completeness: has little effect on the prediction (SHAP=0.0)


In [13]:
top_3 = contribution_df.head(3)

print("Top 3 contributing features:")
print()

for _, row in top_3.iterrows():

    direction = (
        "increases risk"
        if row["shap_value"] > 0
        else "decreases risk"
    )

    print(
        f"• {row['feature']} = {row['value']} "
        f"→ {direction} "
        f"(SHAP: {row['shap_value']:.4f})"
    )

Top 3 contributing features:

• posts = 32.0 → decreases risk (SHAP: -0.1604)
• followers = 1000.0 → decreases risk (SHAP: -0.1425)
• follower_following_ratio = 1.0471204188481675 → decreases risk (SHAP: -0.0917)
